# TechMind — Limpieza de texto

Este notebook continúa el flujo de `05_exploracion_dataset_final.ipynb` y aplica al dataset unificado la misma función que utilizará Backend. El proceso convierte el texto a minúsculas, reemplaza puntuación Unicode, normaliza espacios y elimina stopwords en español de NLTK.

La implementación canónica vive en `shared/limpieza_texto.py`; no se redefine aquí.

## Flujo, preparación y reutilización

Los notebooks se ejecutan en este orden:

1. `05_exploracion_dataset_final.ipynb`: consolida y genera el dataset unificado de entrada.
2. `06_limpieza_texto.ipynb`: aplica la limpieza y genera la columna `texto_limpio`.

Después de instalar las dependencias, el corpus de stopwords se prepara una sola vez por entorno:

```bash
python -m nltk.downloader stopwords
```

Data Science y Backend deben importar la misma función para procesar de forma idéntica los textos de entrenamiento y los textos nuevos:

```python
from shared.limpieza_texto import limpiar_texto

texto_limpio = limpiar_texto("¡Curso práctico de Python para Backend!")
```

## 1. Importaciones y rutas

In [1]:
from pathlib import Path
import sys
import time

import nltk
import pandas as pd

In [2]:
def buscar_raiz_repositorio() -> Path:
    for candidata in (Path.cwd(), *Path.cwd().parents):
        if (candidata / 'shared' / 'limpieza_texto.py').is_file():
            return candidata
    raise FileNotFoundError('No se encontró la raíz del repositorio.')


RAIZ_REPOSITORIO = buscar_raiz_repositorio()
if str(RAIZ_REPOSITORIO) not in sys.path:
    sys.path.insert(0, str(RAIZ_REPOSITORIO))

from shared.limpieza_texto import limpiar_texto

RUTA_ENTRADA = (
    RAIZ_REPOSITORIO
    / 'data_science'
    / 'data'
    / 'procesados'
    / 'dataset_FINAL_UNIFICADO_techmind.csv'
)
RUTA_SALIDA = RUTA_ENTRADA.with_name(
    'dataset_FINAL_UNIFICADO_techmind_limpio.csv'
)

print(f'Entrada: {RUTA_ENTRADA.relative_to(RAIZ_REPOSITORIO)}')
print(f'Salida: {RUTA_SALIDA.relative_to(RAIZ_REPOSITORIO)}')

Entrada: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind.csv
Salida: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind_limpio.csv


## 2. Verificación de recursos y carga del dataset

In [3]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError as error:
    raise RuntimeError(
        'Falta el corpus de stopwords. Ejecuta: '
        'python -m nltk.downloader stopwords'
    ) from error

if not RUTA_ENTRADA.is_file():
    raise FileNotFoundError(f'No existe el dataset de entrada: {RUTA_ENTRADA}')

df = pd.read_csv(RUTA_ENTRADA)
if 'texto' not in df.columns:
    raise ValueError("El dataset no contiene la columna requerida 'texto'.")

print(f'Registros cargados: {len(df):,}')
print(f'Columnas originales: {list(df.columns)}')

Registros cargados: 1,400
Columnas originales: ['titulo', 'texto', 'categoria', 'autor', 'tipo']


## 3. Aplicación de `limpiar_texto()` y exportación

In [4]:
inicio = time.perf_counter()
df['texto_limpio'] = df['texto'].fillna('').map(limpiar_texto)
duracion = time.perf_counter() - inicio

df.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')

print(f'Registros procesados: {len(df):,}')
print(f'Tiempo de limpieza: {duracion:.2f} segundos')
print(
    'Dataset generado: '
    f'{RUTA_SALIDA.relative_to(RAIZ_REPOSITORIO)}'
)

Registros procesados: 1,400
Tiempo de limpieza: 0.25 segundos
Dataset generado: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind_limpio.csv


## 4. Validación del resultado

In [5]:
df_verificacion = pd.read_csv(RUTA_SALIDA)

assert len(df_verificacion) == len(df), 'Cambió el número de registros.'
assert 'texto_limpio' in df_verificacion.columns, 'No se generó texto_limpio.'
assert df_verificacion['texto_limpio'].notna().all(), 'Hay valores nulos en texto_limpio.'

resumen = pd.DataFrame({
    'metrica': [
        'registros',
        'textos originales vacíos',
        'textos limpios vacíos',
        'longitud media original',
        'longitud media limpia',
    ],
    'valor': [
        len(df_verificacion),
        int(df['texto'].fillna('').str.strip().eq('').sum()),
        int(df_verificacion['texto_limpio'].fillna('').str.strip().eq('').sum()),
        round(df['texto'].fillna('').str.len().mean(), 2),
        round(df_verificacion['texto_limpio'].fillna('').str.len().mean(), 2),
    ],
})
resumen

,metrica,valor
0,registros,1400.00
1,textos originales vacíos,0.00
2,textos limpios vacíos,0.00
3,longitud media original,1516.62
4,longitud media limpia,1117.76


In [6]:
pd.set_option('display.max_colwidth', 140)
df_verificacion[['texto', 'texto_limpio']].head(5)

,texto,texto_limpio
0,El sistema de cerraduras de puertas inteligentes basado en el concepto de Internet de las cosas con backend móvil como servicio es el de...,sistema cerraduras puertas inteligentes basado concepto internet cosas backend móvil servicio desarrollo cerraduras puertas inteligentes...
1,"Hay muchas consideraciones que influyen en la ingeniería de un nuevo producto. Independientemente de cuál sea el producto, existen funda...",muchas consideraciones influyen ingeniería nuevo producto independientemente cuál producto existen fundamentos forma ajuste función trat...
2,"En agosto de 2019, organizamos el segundo Seminario de Software de Viena (VSS) con el tema ""DevOps y API de microservicios"". 1 Aprovecha...",agosto 2019 organizamos segundo seminario software viena vss tema devops api microservicios 1 aprovechando recepción positiva primera it...
3,Transmita datos de telemetría operativos simulados de los recursos físicos a Azure Digital Twins y visualice los datos en Unity y un ent...,transmita datos telemetría operativos simulados recursos físicos azure digital twins visualice datos unity entorno realidad mixta aprend...
4,&lt;p&gt;&lt;strong&gt;Resumen&lt;/strong&gt;&lt;/p&gt;&lt;p&gt;Usaha Dagang “Taru Lestari” memiliki kendala di dalam pengelolaan data k...,lt p gt lt strong gt resumen lt strong gt lt p gt lt p gt usaha dagang taru lestari memiliki kendala di dalam pengelolaan data keuangan ...
